# Apprenabilité de la charge : entropie moyenne (Barratt et al. 2022, Fig. 2c)

Issue #115. Ce notebook reproduit la Fig. 2c de F. Barratt, U. Agrawal, A. C. Potter, S. Gopalakrishnan et R. Vasseur, « Transitions in the learnability of global charges from local measurements », PRL 129, 200602 (2022), et la superpose à la courbe numérisée de l'article (`L = 6` et `L = 20` dans l'article, `N = 6` et `N = 20` ici). C'est une validation des fonctions de la bibliothèque (`QELM.models.circuits`, `QELM.evaluation.learnability`) sur un résultat publié.

Le protocole est celui de la Fig. 2b (`notebooks/03_learnability_accuracy.ipynb`, même script `experiments/learnability/learnability.py`, config `conf/learnability.yaml`) :

- une chaîne de `N` qubits (le `L` de l'article) part de la superposition uniforme des chaînes de bits de charge `Q0 = N // 2` (la moitié des relevés) ou `Q1 = N // 2 - 1` (l'autre moitié) ;
- elle évolue pendant `t_f = N` pas d'un circuit en briques de portes U(1) aléatoires (un circuit neuf par relevé), un pas étant une couche paire puis une couche impaire ; après chaque couche, chaque site est mesuré en Z avec la probabilité `p` (règle de Born, `learnability.measure_every=layer`) ;
- Eve voit le relevé mais pas les portes. Son classifieur optimal est le postérieur bayésien `P(Q | m)` calculé avec la dynamique moyennée sur les portes de Haar, un processus d'exclusion symétrique, et `P_corr = P(Q* | m)` est le postérieur de la vraie charge.

L'observable de la Fig. 2c est l'entropie moyenne `E[H(P_corr)]` de la distribution binaire `{P_corr, 1 - P_corr}`, en nats comme dans l'article : `ln 2 ≈ 0.693` quand Eve ne sait rien (`p = 0`), 0 quand elle connaît la charge. C'est le paramètre d'ordre de la transition de succès ; l'encart est le ratio de Binder de `H(P_corr)`, dont les courbes pour plusieurs `N` se croisent vers `p_success ≈ 0.2` dans l'article.

L'article ne dit pas à quel moment d'un pas les sites sont mesurés. Mesurer après chaque couche reproduit la courbe numérisée à 0.02 nats près ; mesurer une seule fois par pas (`measure_every=period`) apprend la charge environ deux fois trop lentement en `p`.


In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so experiments/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Paramètres

Chaque valeur peut être changée ici, ou par la variable d'environnement du même nom (c'est ce que fait le test `tests/test_learnability_entropy_notebook.py` pour un petit balayage). Notation (issue #144) : `N` est toujours la longueur de la chaîne, le `L` de l'article (pas d'ancilla ici), et `N_VALUES` en donne la liste. Les courbes de l'article, nommées `L = 6` dans les exports, sont affichées comme `N = 6` ; notre courbe de même taille est `N6` ; avec plusieurs tailles, la cellule de Binder donne le croisement. `N_SAMPLES` est le nombre de relevés par point (40 000 dans l'article), `samples.L` dans la config. `RUN = False` relit le dernier balayage sans en relancer un.

Durée indicative : environ 10 s par point à `N = 6` et 1 à 2 min par point à `N = 8`.


In [ ]:
N_SAMPLES = os.environ.get("N_SAMPLES", "40000")  # records per run (samples.L)
N_VALUES = os.environ.get("N_VALUES", "6")  # chain lengths N, the paper's L
SEEDS = os.environ.get("SEEDS", "0")
P_VALUES = os.environ.get(
    "P_VALUES",
    "0.0 0.025 0.05 0.075 0.1 0.125 0.15 0.175 0.2 0.25 0.3 0.35 0.4 0.5 0.6",
)
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Lancer le balayage


In [ ]:
from QELM.utils import latest_sweep, run_with_progress

if RUN:
    env = {
        **os.environ,
        "L": N_SAMPLES,  # the launcher reads the sample count from L
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "OUT_ROOT": str(OUT_ROOT),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/learnability/learnability.sh"], total=n_runs, env=env
    )

sweep_dir = latest_sweep(OUT_ROOT / "learnability")
print(f"sweep: {sweep_dir}")

## Relire les métriques

Une ligne par `(N, p)`, moyennée sur les seeds. L'erreur est l'erreur standard sur les seeds, ou l'erreur bootstrap sur les relevés avec une seule seed.


In [ ]:
from QELM.utils import load_sweep, summarize

METRICS = ["entropy_mean", "entropy_var", "entropy_binder", "accuracy"]
rows = load_sweep(
    sweep_dir, keys=(("N", "N"), ("seed", "seed"), ("p", "learnability.p"))
)
table = summarize(
    rows, by=("N", "p"), metrics=[*METRICS, *(f"{m}_err" for m in METRICS)]
)
print(f"{len(rows)} runs")
print(f"{'N':>3} {'p':>6} {'runs':>4} " + " ".join(f"{m:>19}" for m in METRICS))
for r in table:
    errors = [r[f"{m}_sem"] if r["n_runs"] > 1 else r[f"{m}_err_mean"] for m in METRICS]
    values = " ".join(
        f"{r[f'{m}_mean']:>9.4f} ± {e:<7.4f}"
        for m, e in zip(METRICS, errors, strict=True)
    )
    print(f"{r['N']:>3} {r['p']:>6.3f} {r['n_runs']:>4} {values}")

## Superposition à la Fig. 2c

À gauche l'entropie moyenne, avec les courbes numérisées de l'article en tirets (`experiments/learnability/reference/barratt2022_fig2c_entropy.csv`) ; à droite le ratio de Binder de l'entropie (l'encart de la Fig. 2c). La ligne pointillée verticale marque `p_success = 0.2`. `comparison` donne l'écart à la référence de même `N` (aux points de la référence, notre courbe étant interpolée) et le `p` où les courbes de Binder de deux tailles se croisent.


In [ ]:
import importlib.util

import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location(
    "plot_learnability", "experiments/learnability/plot_learnability.py"
)
plot_learnability = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_learnability)

panels, transition, filename, comparison_name = plot_learnability.FIGURES["2c"]
comparison = plot_learnability.plot_sweep(
    sweep_dir,
    plot_learnability.REFERENCE_DIR,
    panels,
    filename,
    transition=transition,
    comparison_name=comparison_name,
)
figure_path = sweep_dir / filename
for metric, crossings in comparison.get("binder_crossings", {}).items():
    for pair, p_cross in crossings.items():
        print(f"{metric} crossing {pair}: {p_cross}")
for metric, by_size in comparison.items():
    if metric == "binder_crossings":
        continue
    for size, curves in by_size.items():
        for name, stats in curves.items():
            print(
                f"{metric} {size} vs {name}: rms = {stats['rms_diff']:.3g}, "
                f"max = {stats['max_abs_diff']:.3g} ({stats['n_points']} points)"
            )

fig = plt.figure(figsize=(11, 4))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
plt.show()